# GOLD ATP PLAYER-MATCH STATS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("fact_player_match_stats").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
# silver
tb_player_match = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

# gold
tb_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_entry = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_entry")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Matches

In [6]:
df = (
    tb_player_match.alias("p_m")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(
        tb_players.alias("pw"),
        (f.col("p_m.PLAYER_ID").cast("string") == f.col("pw.PLAYER_ID").cast("string"))
        | (
            f.col("p_m.PLAYER_ID").cast("string")
            == f.col("pw.PLAYER_ID_OLD").cast("string")
        ),
        "left",
    )
    .join(
        tb_players.alias("po"),
        (f.col("p_m.PLAYER_OPPONENT_ID").cast("string") == f.col("po.PLAYER_ID").cast("string"))
        | (
            f.col("p_m.PLAYER_OPPONENT_ID").cast("string")
            == f.col("po.PLAYER_ID_OLD").cast("string")
        ),
        "left",
    )

    .select(
        f.col("p_m.MATCH_ID"),
        f.col("pw.SK_PLAYER"),
        f.col("po.SK_PLAYER").alias("SK_PLAYER_OPPONENT"),
        f.col("t.SK_TOURNEY"),

        f.col("p_m.PLAYER_IS_WINNER").alias("PLAYER_IS_WINNER"),
        f.coalesce(f.col("p_m.MATCH_DATE").cast("int"), f.lit(-1)).alias("MATCH_DATE"),
        f.coalesce(f.col("p_m.MATCH_NUM").cast("int"), f.lit(0)).alias("MATCH_NUM"),
        f.coalesce(f.col("p_m.MATCH_SCORE").cast("string"), f.lit('-')).alias("MATCH_SCORE"),
        f.coalesce(f.col("p_m.MATCH_BEST_OF").cast("int"), f.lit(3)).alias("MATCH_BEST_OF"),
        f.coalesce(f.col("p_m.MATCH_ROUND").cast("string"), f.lit('-')).alias("MATCH_ROUND"), 
        f.coalesce(f.col("p_m.MATCH_DURATION_M").cast("int"), f.lit(0)).alias("MATCH_DURATION_M"),
        f.coalesce(f.col("p_m.PLAYER_ACES").cast("int"), f.lit(0)).alias("PLAYER_ACES"), 
        f.coalesce(f.col("p_m.PLAYER_DB_FAULTS").cast("int"), f.lit(0)).alias("PLAYER_DB_FAULTS"), 
        f.coalesce(f.col("p_m.PLAYER_SERVE_PTS").cast("int"), f.lit(0)).alias("PLAYER_SERVE_PTS"), 
        f.coalesce(f.col("p_m.PLAYER_1ST_SERVES_IN").cast("int"), f.lit(0)).alias("PLAYER_1ST_SERVES_IN"), 
        f.coalesce(f.col("p_m.PLAYER_1ST_SERVE_PTS_WON").cast("int"), f.lit(0)).alias("PLAYER_1ST_SERVE_PTS_WON"), 
        f.coalesce(f.col("p_m.PLAYER_2ND_SERVE_PTS_WON").cast("int"), f.lit(0)).alias("PLAYER_2ND_SERVE_PTS_WON"), 
        f.coalesce(f.col("p_m.PLAYER_SERVE_GAMES").cast("int"), f.lit(0)).alias("PLAYER_SERVE_GAMES"), 
        f.coalesce(f.col("p_m.PLAYER_BP_SAVED").cast("int"), f.lit(0)).alias("PLAYER_BP_SAVED"), 
        f.coalesce(f.col("p_m.PLAYER_BP_FACED").cast("int"), f.lit(0)).alias("PLAYER_BP_FACED") 
    )
    .distinct()
)

## Validade final dataframe

In [7]:
if tb_player_match.count() == df.count():
    print('ok')
else: 
    raise

ok


## Save dataframe

### Local

In [8]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_match_stats.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)

Py4JJavaError: An error occurred while calling o191.save.
: org.postgresql.util.PSQLException: FATAL: (ENOTFOUND) tenant/user postgres.iegwyopgfymnkedaqdos not found
	at org.postgresql.core.v3.ConnectionFactoryImpl.doAuthentication(ConnectionFactoryImpl.java:837)
	at org.postgresql.core.v3.ConnectionFactoryImpl.tryConnect(ConnectionFactoryImpl.java:309)
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:367)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:52)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:295)
	at org.postgresql.Driver.makeConnection(Driver.java:437)
	at org.postgresql.Driver.connect(Driver.java:305)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.s